# 12.1 Concurrency, Parallelism and the GIL

**Prerequisites:** 04 Functions, 06 Exception Handling, 11.4 (which used threads and asyncio informally)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 **Concurrency is not parallelism** - the distinction everything else rests on
- CPU-bound vs I/O-bound, and how to tell which you have
- **What the GIL actually is**, and the three myths about it
- Measuring it: threads on CPU work, threads on I/O work
- Processes, threads and `async` - what each costs
- > **Version note:** the free-threaded build (PEP 703) in 3.13/3.14
- A decision guide for the rest of the folder

---

## 🔴 Concurrency is not parallelism

These get used interchangeably and they are different ideas.

> **Concurrency** is *dealing with* many things at once — structuring a program so several tasks are in progress.
>
> **Parallelism** is *doing* many things at once — several tasks literally executing at the same instant, on different cores.

### The kitchen

One cook with a stew simmering, bread proving and vegetables to chop is **concurrent**: three jobs in progress, one pair of hands, switching whenever something needs attention. Nothing happens simultaneously — but the meal is ready far sooner than doing each job start-to-finish.

Three cooks working at once is **parallel**.

```
  concurrent, not parallel      A─B─A─B─A─B──>   one core, interleaved
  parallel                      A─A─A─A─A──>     two cores, simultaneous
                                B─B─B─B─B──>
```

Concurrency is a way of **structuring** a program. Parallelism is a way of **executing** it. You can have either without the other.

This matters here because **Python gives you concurrency easily and parallelism only with care** — and the reason is the GIL.

## First: what kind of work is it?

Every optimisation decision in this folder starts here.

| | **CPU-bound** | **I/O-bound** |
|---|---|---|
| Spends its time | executing instructions | **waiting** |
| Examples | parsing, compression, hashing, image work, numeric loops | HTTP requests, database queries, file reads, `sleep` |
| Faster with more cores? | yes | no - it was never busy |
| Right tool | **processes** | **threads** or **async** |

### How to tell

Watch a CPU core while it runs. Pinned at 100%? CPU-bound. Near idle while the program takes ten seconds? It is waiting, and that wait is your opportunity.

> **Most application slowness is I/O-bound**, which is good news: waiting is the easy case. Ten HTTP requests taking one second each take one second total if you overlap the waiting, and no extra cores are needed.

## The GIL, accurately

The **Global Interpreter Lock** is a single lock inside CPython. To execute Python bytecode, a thread must hold it. There is one, so:

> **Only one thread executes Python bytecode at a time, per interpreter.**

It exists because CPython's memory management (reference counting) is not thread-safe. One lock around the interpreter is simple and makes single-threaded code fast; fine-grained locking would be neither.

### The three myths

| Myth | Reality |
|---|---|
| "Threads are useless in Python" | False. A thread **releases the GIL while waiting on I/O**, which is why threaded network code works (**11.4**). |
| "The GIL makes Python slow" | It makes *multi-threaded CPU-bound* code no faster. Single-threaded speed is unaffected — arguably helped. |
| "You can never use multiple cores" | You can: **processes** each have their own GIL. C extensions like NumPy also release it during heavy work. |

### The one sentence to keep

```
    The GIL is released while a thread waits.
    It is held while a thread computes.
```

So: threads help when your threads **wait**, and do nothing when they **compute**. Everything below measures exactly that.

In [ ]:
import sys
import sysconfig
import threading
import time
from concurrent.futures import ThreadPoolExecutor

print("Python          :", sys.version.split()[0])
print("free-threaded   :", bool(sysconfig.get_config_var("Py_GIL_DISABLED")))
gil_probe = getattr(sys, "_is_gil_enabled", None)
print("GIL enabled     :", gil_probe() if gil_probe else "(no probe on this build)")
print("switch interval :", sys.getswitchinterval(), "seconds")
print()
print("The switch interval is how long a CPU-bound thread may hold the GIL")
print("before being asked to give it up. It is why threads interleave at all.")

### Measurement 1: CPU-bound work on threads

Counting primes with a plain Python loop — no I/O, no library that might release the GIL. If threads helped CPU-bound work, four threads would be roughly four times faster.

Watch what actually happens.

In [ ]:
def count_primes(limit):
    """Deliberately naive and pure Python: this is CPU-bound."""
    found = 0
    for candidate in range(2, limit):
        divisor, is_prime = 2, True
        while divisor * divisor <= candidate:
            if candidate % divisor == 0:
                is_prime = False
                break
            divisor += 1
        if is_prime:
            found += 1
    return found


LIMIT = 120_000
TASKS = 4

started = time.perf_counter()
serial_result = [count_primes(LIMIT) for _ in range(TASKS)]
cpu_serial = time.perf_counter() - started
print(f"serial  : {cpu_serial:5.2f}s   ({serial_result[0]:,} primes each)")

started = time.perf_counter()
with ThreadPoolExecutor(max_workers=TASKS) as pool:
    list(pool.map(count_primes, [LIMIT] * TASKS))
cpu_threaded = time.perf_counter() - started
print(f"threads : {cpu_threaded:5.2f}s   speedup {cpu_serial / cpu_threaded:.2f}x")

print()
if cpu_threaded >= cpu_serial * 0.9:
    print("🔴 No speedup - and quite possibly slower. Four threads took turns")
    print("   holding one GIL, and paid for the switching on top.")
else:
    print("Some speedup - this build may be free-threaded, or the work leaked")
    print("into C code that releases the GIL.")
print(f"\n   {TASKS} threads, {sys.version_info.major}.{sys.version_info.minor}, "
      f"expected ~{TASKS}x if the GIL were not there.")

### Measurement 2: I/O-bound work on threads

Identical structure, but each task **waits** instead of computing. `time.sleep()` releases the GIL — as does any real I/O: a socket read, a database query, a file read.

This is the same code shape as the last cell. Only the nature of the work changed.

In [ ]:
def wait_for_io(seconds):
    """Stands in for a network call or a database query."""
    time.sleep(seconds)          # releases the GIL while waiting
    return seconds


DELAY = 0.4

started = time.perf_counter()
for _ in range(TASKS):
    wait_for_io(DELAY)
io_serial = time.perf_counter() - started
print(f"serial  : {io_serial:5.2f}s   ({TASKS} x {DELAY}s of waiting)")

started = time.perf_counter()
with ThreadPoolExecutor(max_workers=TASKS) as pool:
    list(pool.map(wait_for_io, [DELAY] * TASKS))
io_threaded = time.perf_counter() - started
print(f"threads : {io_threaded:5.2f}s   speedup {io_serial / io_threaded:.2f}x")

print()
print("✅ Nearly linear. The GIL was released by every sleeping thread, so")
print("   all four waited at the same time.")
print()
print("Same threads. Same pool. Same code shape. Opposite result -")
print("and the ONLY difference is whether the work waits or computes.")

### The two results side by side

This is the whole notebook in one table, produced by the two cells above.

In [ ]:
print(f"{'workload':<14}{'serial':>9}{'threaded':>10}{'speedup':>10}")
print("-" * 43)
print(f"{'CPU-bound':<14}{cpu_serial:>8.2f}s{cpu_threaded:>9.2f}s"
      f"{cpu_serial / cpu_threaded:>9.2f}x")
print(f"{'I/O-bound':<14}{io_serial:>8.2f}s{io_threaded:>9.2f}s"
      f"{io_serial / io_threaded:>9.2f}x")
print()
print("Threads are not a general-purpose speed-up. They are a tool for")
print("overlapping WAITING. For CPU work you need processes - see 12.3.")

## The three models

| | **Threads** | **Processes** | **`async`** |
|---|---|---|---|
| Module | `threading` | `multiprocessing` | `asyncio` |
| Memory | shared | separate | shared |
| Parallel on cores? | 🔴 no (GIL) | ✅ yes | 🔴 no |
| Good for | I/O-bound | CPU-bound | **lots** of I/O-bound |
| Cost to start one | ~50 µs | ~50 ms on Windows | ~1 µs |
| Practical count | hundreds | one per core | tens of thousands |
| Switching | pre-emptive - anywhere | n/a | **cooperative** - only at `await` |
| Data sharing | direct, needs locks | pickled, explicit | direct, no locks needed |
| Main hazard | 🔴 race conditions | startup cost, pickling | 🔴 one blocking call stalls all |

### Pre-emptive vs cooperative — the practical difference

A thread can be interrupted **between any two bytecodes**, so any read-modify-write on shared state is unsafe by contract — see **12.2**, where a single yield point between the read and the write loses 75% of the updates. (Interestingly, the *tight* `counter += 1` loop does not lose anything on CPython 3.14 — but that is an implementation accident, not a promise.) An `async` task yields **only at an `await`**, so between awaits you have the machine to yourself and need no lock.

That trade is the heart of it: `asyncio` removes a whole category of bug, and in exchange every blocking call becomes a landmine (**11.4** measured one costing 5.8x).

> ### Version note - the free-threaded build (PEP 703)
>
> **3.13** introduced an official *free-threaded* build of CPython with **no GIL**, and **3.14** improved it further and made it officially supported rather than experimental.
>
> It is a **separate build** (`python3.13t` / `python3.14t`), not a flag on a normal install. The cell near the top of this notebook reports whether you are on one.
>
> | | Standard build | Free-threaded build |
> |---|---|---|
> | CPU-bound threading | no gain | real parallelism |
> | Single-threaded speed | baseline | a few % slower |
> | C extension support | universal | growing, not universal |
> | Race conditions | still possible | **more likely to bite** |
>
> 🔴 Removing the GIL does **not** make threaded code safe. It removes an accidental protection: code that got away with unsynchronised access because the GIL happened to make some operations atomic can now genuinely corrupt data. Everything **12.2** says about locks matters more, not less.
>
> For now: write as though the GIL exists, use processes for CPU-bound work, and treat free-threading as a future optimisation rather than today's plan.

## Choosing - the guide for this folder

```
  Is the work CPU-bound or I/O-bound?
  │
  ├─ CPU-bound ──> multiprocessing / ProcessPoolExecutor        12.3, 12.4
  │                (or NumPy etc., which release the GIL)
  │
  └─ I/O-bound ──> how many concurrent operations?
                   │
                   ├─ a few dozen ────> threads                 12.2, 12.4
                   │                    simplest; blocking libraries fine
                   │
                   └─ hundreds+ ──────> asyncio                 12.5
                                        needs async-aware libraries
```

### Two honest warnings

**Measure before you start.** Concurrency adds real complexity and a class of bug that only appears under load. A single-threaded program you can reason about beats a concurrent one you cannot.

**The fastest fix is often not concurrency at all** — a database index, a cache, or a better algorithm (**15**) routinely beats parallelising the slow version.

| Next | Covers |
|---|---|
| **12.2** | `threading`: races, locks, deadlock, queues |
| **12.3** | `multiprocessing`: real parallelism, and the `spawn` guard |
| **12.4** | `concurrent.futures`: one API over both |
| **12.5** | `asyncio`: coroutines, tasks, `TaskGroup` |

---

## Common Mistakes & Pitfalls

1. 🔴 **Using threads to speed up CPU-bound work.** The GIL means they take turns; you get no gain and pay for switching.
2. 🔴 **Believing the GIL makes threads pointless.** It is released during I/O, which is what most programs actually wait on.
3. **Not establishing whether the work is CPU- or I/O-bound before choosing.** That one question determines the answer.
4. **Assuming the free-threaded build makes threading safe.** It removes an accidental protection - races become *more* likely, not less.
5. **Reaching for concurrency before measuring.** An index or a cache is often a bigger win with none of the risk.
6. **Comparing against an unmeasured baseline.** Without the serial timing, a 'speedup' is a guess.
7. **Assuming `asyncio` is simply faster.** It is a different structure, not a faster one - and it is slower than threads if your libraries block.

## Best Practices

- Classify the workload first: CPU-bound or I/O-bound.
- Measure serial, then concurrent, on the same machine and workload.
- Use processes for CPU-bound work, threads or `asyncio` for I/O-bound.
- Prefer `concurrent.futures` (**12.4**) over raw `Thread`/`Process` objects.
- Keep the concurrent part small and the shared state smaller.
- Write code that is correct with and without the GIL - do not rely on accidental atomicity.
- State in a comment why a piece of code is concurrent; the next reader cannot tell from the code alone.

## Practice Exercises

Try these before moving on.

1. Raise `TASKS` to 8 and re-run both measurements. Does the CPU-bound result get worse? Why would more threads make it *slower* rather than merely not faster?
2. Replace `count_primes` with `hashlib.sha256` over a large `bytes` object and re-run the threaded test. `hashlib` releases the GIL - does the result change?
3. Set `sys.setswitchinterval(0.0001)` and re-run the CPU-bound test. What happens, and what does that tell you about the cost of switching?
4. Time how long `threading.Thread` takes to start, versus `multiprocessing.Process`, on your machine. Explain the difference.
5. Take a slow script of your own and work out whether it is CPU- or I/O-bound - by watching CPU usage, and by timing with `time.process_time()` against `time.perf_counter()`.
6. 🔴 Predict, before running: three threads each doing 1s of `time.sleep` and one thread doing 1s of prime counting. How long in total, and why?